# Bitta HLS tile bo'yicha oylik ET (VIIRS) — xaritali qadamlar

Har qadam **geemap** xaritasiga qo'shiladi. Barcha mantiq tayyor modullardan (`sebal_gee_v4`) chaqiriladi — cell ichida kod qayta yozilmaydi.

Tile: **T41SPD** (Qashqadaryo) · Mart/May 2026 · `pysebal`.

In [2]:
# Cell 1 — Setup va importlar
import ee, geemap
ee.Initialize(project='carbon-science-461016-q2')

from sebal_gee_v4 import ee_utils; ee_utils.install_getinfo_retry()
from sebal_gee_v4 import main as M
from sebal_gee_v4 import viirs_downscaling as vds
print('OK')

OK


In [3]:
# Cell 2 — Konfiguratsiya + TILE chegarasi
TILE   = 'T41SQD'
SAT    = 'HLS'
START, END = '2026-05-01', '2026-05-31'
MODE   = 'lambda'      # 'lambda' (EVAP_FRAC) yoki 'kc' (KC)
MODEL  = 'multi'       # 'ndvi' | 'ndvi2' | 'multi'
QA     = 'lenient'

roi = (ee.FeatureCollection('FAO/GAUL/2015/level1')
       .filter(ee.Filter.eq('ADM1_NAME', 'Kashkadarya')).geometry())

# TILE geometriyasi — endi FAQAT shu chegarada ishlaymiz (429 yo'q)
tile_geom = M.get_hls_tile_geometry(TILE, START, END)
tile_roi  = roi.intersection(tile_geom, ee.ErrorMargin(30))

Map = geemap.Map()
Map.centerObject(tile_roi, 9)
Map.addLayer(roi, {'color': 'gray'}, 'Viloyat ROI', True, 0.2)
Map.addLayer(tile_roi, {'color': 'red'}, f'TILE {TILE} chegarasi', True, 0.5)
Map

Map(center=[39.06981041602096, 65.82651274482228], controls=(WidgetControl(options=['position', 'transparent_b…

In [4]:
# Cell 3 — Sahnalarni topish (process_tile) + NDVI xaritada
scenes, info = M.process_tile(roi, START, END, 'pysebal', SAT, 70, TILE)
anchors = [{'image': ee.Image(s), 'date': info['dates'][i]}
           for i, s in enumerate(scenes)]
print('Sahnalar:', len(anchors), '| sanalar:', info['dates'])

ndvi_vis = {'min': 0, 'max': 0.8, 'palette': ['white', 'khaki', 'green']}
Map.addLayer(anchors[0]['image'].select('NDVI'), ndvi_vis,
             f"NDVI {anchors[0]['date']}")
Map

      ✅ 2026-05-08: ekin bulut 7.4%
      ✅ 2026-05-15: ekin bulut 12.7%
      ✅ 2026-05-16: ekin bulut 3.5%
  [T41SQD] Filtrlangan: Path=None Row=None → 3 tasvir
  [T41SQD] Tasvirlar: 3 | ['2026-05-08', '2026-05-15', '2026-05-16']
  [T41SQD] Sahna 1/3...
  🌾 Anchor cropland mask pixel count: {'SLOPE': 2498849}
  [T41SQD] Sahna 2/3...
  🌾 Anchor cropland mask pixel count: {'SLOPE': 2498849}
  [T41SQD] Sahna 3/3...
  🌾 Anchor cropland mask pixel count: {'SLOPE': 2498849}
Sahnalar: 3 | sanalar: ['2026-05-08', '2026-05-15', '2026-05-16']


Map(bottom=50360.0, center=[39.06981041602096, 65.82651274482228], controls=(WidgetControl(options=['position'…

In [5]:
# Cell 4 — Per-scene SEBAL natijalari (EVAP_FRAC, ET_24, KC)
ef_vis = {'min': 0, 'max': 1, 'palette': ['red', 'yellow', 'green', 'blue']}
et_vis = {'min': 0, 'max': 8, 'palette': ['white', 'cyan', 'blue', 'navy']}
kc_vis = {'min': 0, 'max': 1.2, 'palette': ['brown', 'yellow', 'green']}

a0 = anchors[0]
Map.addLayer(a0['image'].select('EVAP_FRAC'), ef_vis, f"Λ EVAP_FRAC {a0['date']}")
Map.addLayer(a0['image'].select('ET_24'), et_vis, f"ET_24 {a0['date']}")
Map.addLayer(a0['image'].select('KC'), kc_vis, f"KC {a0['date']}", False)
Map

Map(bottom=50360.0, center=[39.06981041602096, 65.82651274482228], controls=(WidgetControl(options=['position'…

In [6]:
# Cell 5 — VIIRS vegetation predictor (NDVI) bitta kun uchun
vproj = vds.get_viirs_projection(vds.get_viirs_vnp09ga(START, roi))
preds0 = vds.clear_viirs_predictors(a0['date'], roi, vproj, QA)
Map.addLayer(preds0.select('NDVI').clip(roi), ndvi_vis,
             f"VIIRS NDVI {a0['date']}")
Map

Map(bottom=50360.0, center=[39.06981041602096, 65.82651274482228], controls=(WidgetControl(options=['position'…

In [8]:
# Cell 6 — Regressiya (VIIRS predictor → Λ). 30m namunalardan fit.
for a in anchors:
    a['vproj'] = vproj
fc = vds.build_regression_samples_30m(anchors, roi, MODE, QA)
snp = vds.samples_to_numpy(fc)
reg = vds.fit_viirs_regression(snp, MODEL)
print('Regressiya:', {k: reg[k] for k in ['R2','RMSE','MAE','N']})
print('Koeffitsientlar:', reg['coeffs'])

  ⏳ GEE band (Too many concurrent aggregations.). 10s kutilmoqda... [urinish 1/6]


KeyboardInterrupt: 

In [7]:
# Cell 7 — Fazoviy weight: W = Λ_landsat / predict(VIIRS)
tband = 'EVAP_FRAC' if MODE == 'lambda' else 'KC'
W = vds.weight_from_prediction(a0['image'].select(tband), preds0,
                               reg, MODEL, MODE)
w_vis = {'min': 0.2, 'max': 2.0, 'palette': ['blue', 'white', 'red']}
Map.addLayer(W.clip(roi), w_vis, 'Weight (W≈1)')
Map

NameError: name 'reg' is not defined

In [ ]:
# Cell 8 — Bitta VIIRS kun: Λ_30 = predict × W, va kunlik ET
TEST_DAY = '2026-05-18'   # Landsat yo'q kun (VIIRS bilan)
preds_d = vds.clear_viirs_predictors(TEST_DAY, roi, vproj, QA)
lam_d = vds.predict_coarse_target(preds_d, reg, MODEL, MODE)
lam30 = lam_d.multiply(W).rename('LAMBDA_30')
Map.addLayer(lam30.clip(roi), ef_vis, f'Λ_30 {TEST_DAY}')

alb, tau = vds.interp_radiation_bands(anchors, TEST_DAY)
rn24 = vds.daily_rn24(TEST_DAY, roi, alb, tau)
et_d = vds.compute_daily_et_lambda_mode(lam30, rn24)
Map.addLayer(et_d.clip(roi), et_vis, f'ET_24 {TEST_DAY} (VIIRS)')
Map

In [ ]:
# Cell 9 — TILE uchun OYLIK ET (to'liq orkestrator)
monthly = vds.build_tile_monthly_et_viirs(
    scenes, info, roi, START, END, MODE, MODEL, QA, 'linear')

etm_vis = {'min': 0, 'max': 180,
           'palette': ['white', 'yellow', 'green', 'blue', 'navy']}
Map.addLayer(monthly.clip(roi), etm_vis, 'VIIRS OYLIK ET (mm/oy)')
print('VIIRS R2:', monthly.get('viirs_r2').getInfo())
Map

In [ ]:
# Cell 10 — (ixtiyoriy) Kichik hududda oylik ET statistikasi
pt = roi.centroid(100).buffer(8000)
print('Oylik ET (markaz, mm/oy):', monthly.reduceRegion(
    ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    pt, 100, maxPixels=1e9, bestEffort=True).getInfo())